In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_contact_details_arotron(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "ContactDetailId	String"
        #     ",SourceSystem	String"
        #     ",PortfolioKey	String"
        #     ",PortfolioName	String"
        #     ",IngestionTimestamp timestamp"
        #     ",BatchId	String"
        #     ",ContactDetailKey	String"
        #     ",CDI	string"
        #     ",PartyKey	String"
        #     ",ContactDetailTypeCode	string"
        #     ",ContactDetailType	string"
        #     ",ContactDetailTypeGroup string"
        #     ",FullAddress	string"
        #     ",Country	string"
        #     ",County	string"
        #     ",City	string"
        #     ",PostalCode	string"
        #     ",Street	string"
        #     ",Street1 string"
        #     ",Street2 string"
        #     ",Street3 string"
        #     ",StreetNumber	string"
        #     ",PhoneNumber1	string"
        #     ",PhoneNumber2	string"
        #     ",PhoneNumber3	string"
        #     ",EmailAddress1	string"
        #     ",EmailAddress2	string"
        #     ",IsValidFlag	boolean"
        #     ",CommunicationFlag	string"
        #     ",IsPrimaryFlag	boolean"
        #     ",CreatedBy	string"
        #     ",CreatedDate	timestamp"
        #     ",LastUpdatedDate	date"
        #     ",OriginalTelephoneCode	string"
        #     ",OriginalTelephoneType	string"
        #     ",IsActiveFlag	string"
        #     ",DoNotCallFlag	boolean"
        #     ",IsDialerActivityFlag	int"
        #     ",DialerActivityTelephoneNumber	string"
        #     ",InternalConfirmationTypeCode	int"
        #     ",InternalConfirmationType	string"
        #     ",CollectionSourceTypeCode	int"
        #     ",CollectionSourceType	string"
        #     ",DoNotSendFlag boolean"
        #     ",AddContactFlag boolean"
        #     ",SerialNumber int"
        #     ",BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.ContactDetails"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                    spark,
                    curated_table_path,
                    run_date,  # type: ignore
                    "arotron",
                )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")


        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = ["PortfolioName", "Prefix", "unit_code_profits"]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )


        # ================== eaisig_poc ==================

        eaisig_poc_extra_filters = None

        if eaisig_poc_extra_filters is not None:
            eaisig_poc_extra_filters.append(date_filter)
            eaisig_poc_final_filters = " AND ".join(eaisig_poc_extra_filters)
        else:
            eaisig_poc_final_filters = date_filter

        eaisig_poc_selected_fields = [
            "rowid",
            "eaisig_poc_cust_s_unid",
            "eaisig_poc_add_type",
            "eaisig_poc_add_area",
            "eaisig_poc_add_country",
            "eaisig_poc_add_city",
            "eaisig_poc_add_zip",
            "eaisig_poc_add_street1",
            "eaisig_poc_add_street2",
            "eaisig_poc_add_street3",
            "eaisig_poc_add_nuim",
            "eaisig_poc_add_phone1",
            "eaisig_poc_add_phone2",
            "eaisig_poc_add_phone3",
            "eaisig_poc_add_email",
            "eaisig_poc_createdt",
            "eaisig_poc_mod_date",
            "eaisig_poc_add_status",
            "businessdate",
        ]

        eaisig_poc = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_eaisig_poc",
            eaisig_poc_final_filters,
            eaisig_poc_selected_fields,
        )

        # =============== Join with portfolio ===============
        result_df = eaisig_poc.join(
            portfolio,
            on=substring(eaisig_poc["eaisig_poc_cust_s_unid"], 1, 3) == portfolio["Prefix"],
            how="left"
        )

        # HashKeys
        eaisig_poc_primary_key_fields = [
            eaisig_poc.rowid,
            lit("arotron"),
        ]

        result_df = result_df.withColumn(
            "ContactDetailID",
            sha2(concat_ws("", *eaisig_poc_primary_key_fields), 256),
        )

        # ================== Selection of Final Columns ==================

        result_df = result_df.select(
            result_df.ContactDetailID,
            lit("arotron").alias("SourceSystem"),
            coalesce(portfolio.unit_code_profits, lit("-9998")).cast("string").alias("PortfolioKey"),
            coalesce(portfolio.PortfolioName, lit("Core Wholesale")).cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            eaisig_poc.rowid.cast("string").alias("ContactDetailKey"),
            result_df.eaisig_poc_cust_s_unid.alias("CDI"),
            result_df.eaisig_poc_cust_s_unid.alias("PartyKey"),
            result_df.eaisig_poc_add_type.alias("ContactDetailTypeCode"),
            result_df.eaisig_poc_add_type.alias("ContactDetailType"),
            lit(None).alias("ContactDetailTypeGroup"),
            lit(None).alias("FullAddress"),
            result_df.eaisig_poc_add_country.alias("Country"),
            result_df.eaisig_poc_add_area.alias("County"),
            result_df.eaisig_poc_add_city.alias("City"),
            result_df.eaisig_poc_add_zip.alias("PostalCode"),
            result_df.eaisig_poc_add_street1.alias("Street"),
            result_df.eaisig_poc_add_street1.alias("Street1"),
            result_df.eaisig_poc_add_street2.alias("Street2"),
            result_df.eaisig_poc_add_street3.alias("Street3"),
            result_df.eaisig_poc_add_nuim.alias("StreetNumber"),
            result_df.eaisig_poc_add_phone1.alias("PhoneNumber1"),
            result_df.eaisig_poc_add_phone2.alias("PhoneNumber2"),
            result_df.eaisig_poc_add_phone3.alias("PhoneNumber3"),
            result_df.eaisig_poc_add_email.alias("EmailAddress1"),
            lit(None).alias("EmailAddress2"),
            lit(None).alias("IsValidFlag"),
            lit(None).alias("CommunicationFlag"),
            lit(None).alias("IsPrimaryFlag"),
            lit(None).alias("CreatedBy"),
            result_df.eaisig_poc_createdt.alias("CreatedDate"),
            result_df.eaisig_poc_mod_date.cast("date").alias("LastUpdatedDate"),
            lit(None).alias("OriginalTelephoneCode"),
            lit(None).alias("OriginalTelephoneType"),
            result_df.eaisig_poc_add_status.alias("IsActiveFlag"),
            lit(None).alias("DoNotCallFlag"),
            lit(None).alias("IsDialerActivityFlag"),
            lit(None).alias("DialerActivityTelephoneNumber"),
            lit(None).alias("InternalConfirmationTypeCode"),
            lit(None).alias("InternalConfirmationType"),
            lit(None).alias("CollectionSourceTypeCode"),
            lit(None).alias("CollectionSourceType"),
            lit(None).alias("DoNotSendFlag"),
            lit(None).alias("AddContactFlag"),
            lit(None).alias("SerialNumber"),
            result_df.businessdate.alias("BusinessDate"),
        )

        # ================== Append ==================

        result_df.write.format("delta").clusterBy("BusinessDate").mode("append").saveAsTable(
            curated_table_path
        )
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_contact_details_arotron(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)